# Setup 


In [2]:
import os
import json
import numpy as np
import pandas as pd
from collections import defaultdict

DATA_DIR = os.environ.get("KPI_RAG_DATA", "/home/jovyan/kpi_rag/data")

# Load raw data and find session/time columns

In [3]:
df = pd.read_parquet(os.path.join(DATA_DIR, "telecomts_full.parquet"))
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:\n", df.dtypes)
print("\nFirst 3 rows:")
print(df.head(3))
candidate_cols = [c for c in df.columns if any( kw in c.lower() for kw in ["session", "recording", "record", "time", "step", "seq", "id"] )]
print("\nCandidate session/time columns:", candidate_cols)

Shape: (32000, 11)

Columns: ['start_time', 'end_time', 'sampling_rate', 'KPIs', 'description', 'anomalies', 'statistics', 'labels', 'QnA', 'anomaly_type', 'anomaly_status']

Dtypes:
 start_time        datetime64[us]
end_time          datetime64[us]
sampling_rate              int64
KPIs                         str
description                  str
anomalies                    str
statistics                   str
labels                       str
QnA                          str
anomaly_type                 str
anomaly_status               str
dtype: object

First 3 rows:
               start_time                end_time  sampling_rate  \
0 2025-01-01 00:05:33.000 2025-01-01 00:05:45.700             10   
1 2025-01-01 00:12:18.300 2025-01-01 00:12:31.000             10   
2 2025-01-01 00:04:53.000 2025-01-01 00:05:05.700             10   

                                                KPIs  \
0  {'RSRP': array([-73.  , -73.  , -73.  , -73.  ...   
1  {'RSRP': array([-73., -73., -73., -7

# Detect session boundaries via gaps

In [4]:
df_sorted = df.sort_values("start_time").reset_index()  # keep original index as 'index' column
df_sorted["gap_to_next"] = ( df_sorted["start_time"].shift(-1) - df_sorted["end_time"] ).dt.total_seconds()

print("Gap statistics (seconds):")
print(df_sorted["gap_to_next"].describe())
print("\nLargest 40 gaps (likely session boundaries):")
print(df_sorted["gap_to_next"].nlargest(40).values)

for threshold in [1, 5, 10, 30, 60, 300, 3600]:
    n_sessions = (df_sorted["gap_to_next"] > threshold).sum() + 1
    print(f"threshold={threshold}s -> {n_sessions} sessions")

Gap statistics (seconds):
count    3.199900e+04
mean     1.208370e+04
std      8.258156e+05
min     -1.260000e+01
25%     -9.500000e+00
50%     -9.500000e+00
75%     -9.500000e+00
max      9.192868e+07
Name: gap_to_next, dtype: float64

Largest 40 gaps (likely session boundaries):
[9.19286785e+07 8.95094081e+07 5.38263969e+07 2.96343201e+07
 2.66103521e+07 1.87479233e+07 1.57238017e+07 1.20860033e+07
 6.04702410e+06 3.61880330e+06 3.61880330e+06 2.41833610e+06
 2.40920650e+06 2.40920330e+06 2.40920330e+06 2.40920330e+06
 2.40920330e+06 2.24540850e+06 1.81340170e+06 1.81340170e+06
 1.81340170e+06 1.81340170e+06 1.81340170e+06 1.81340170e+06
 1.81340170e+06 1.81340170e+06 1.81340170e+06 1.20860170e+06
 1.20860170e+06 8.54325000e+04 7.64293000e+04 4.06500000e+02
 2.72100000e+02 1.88900000e+02 1.79300000e+02 1.60100000e+02
 1.56900000e+02 1.47300000e+02 1.31300000e+02 1.08900000e+02]
threshold=1s -> 95 sessions
threshold=5s -> 92 sessions
threshold=10s -> 87 sessions
threshold=30s -> 73 se

# Assign session IDs

In [5]:
best_threshold = 300  # 5 minutes — gives 33 sessions, clear gap in the data

df_sorted["session_id"] = (df_sorted["gap_to_next"].shift(1) > best_threshold).cumsum()
df_sorted["session_id"] = df_sorted["session_id"].fillna(0).astype(int)

session_id_by_row = df_sorted.set_index("index")["session_id"].sort_index().values

print("Total sessions found:", len(np.unique(session_id_by_row)))
print("Rows per session:\n", np.bincount(session_id_by_row))

anom_mask_full = (df["anomaly_status"] == "Anomaly").values
anom_sessions = np.unique(session_id_by_row[anom_mask_full])
print("\nSessions containing anomalies:", anom_sessions)
print("Count of anomalous sessions:", len(anom_sessions))
print("Total anomalous rows:", anom_mask_full.sum(), "(should be 1,235)")

for s in anom_sessions:
    n_in_session = (session_id_by_row == s).sum()
    n_anom_in_session = (anom_mask_full & (session_id_by_row == s)).sum()
    print(f" Session {s}: {n_anom_in_session} anomalies / {n_in_session} total windows")

Total sessions found: 33
Rows per session:
 [3100   94   92 3006 3006 3007 3006 3006 3007 3006 3006  147   98   69
  125   99  136  137   59    7  309  309  309  309  309  309  309  309
  309  309  309  309   79]

Sessions containing anomalies: [ 0  1  2 11 12 13 14 15 16 17 18 19 32]
Count of anomalous sessions: 13
Total anomalous rows: 1235 (should be 1,235)
 Session 0: 93 anomalies / 3100 total windows
 Session 1: 94 anomalies / 94 total windows
 Session 2: 92 anomalies / 92 total windows
 Session 11: 147 anomalies / 147 total windows
 Session 12: 98 anomalies / 98 total windows
 Session 13: 69 anomalies / 69 total windows
 Session 14: 125 anomalies / 125 total windows
 Session 15: 99 anomalies / 99 total windows
 Session 16: 136 anomalies / 136 total windows
 Session 17: 137 anomalies / 137 total windows
 Session 18: 59 anomalies / 59 total windows
 Session 19: 7 anomalies / 7 total windows
 Session 32: 79 anomalies / 79 total windows


# Fault-type breakdown per session

In [6]:
anomaly_type_full = df["anomaly_type"].values

print("Fault type composition per anomalous session:")
print("-" * 70)
for s in anom_sessions:
    mask = (session_id_by_row == s) & anom_mask_full
    types_in_session = pd.Series(anomaly_type_full[mask]).value_counts()
    print(f"\nSession {s} ({mask.sum()} rows): {dict(types_in_session)}")

Fault type composition per anomalous session:
----------------------------------------------------------------------

Session 0 (93 rows): {'Jamming': np.int64(93)}

Session 1 (94 rows): {'Jamming': np.int64(94)}

Session 2 (92 rows): {'Jamming': np.int64(92)}

Session 11 (147 rows): {'Co-Channel Interference (Severe)': np.int64(50), 'Buffer Overflow (Gradual Buildup)': np.int64(21), 'Doppler Shift (Severe)': np.int64(18), 'Faulty RF Filters (Temporal)': np.int64(17), 'Co-Channel Interference (Mild)': np.int64(16), 'Resource Allocation Bugs': np.int64(10), 'Faulty Handover Algorithm (Too Frequent)': np.int64(9), 'Antenna Failure': np.int64(6)}

Session 12 (98 rows): {'Antenna Failure': np.int64(27), 'Co-Channel Interference (Mild)': np.int64(25), 'High Network Congestion (Sudden Spike)': np.int64(14), 'High Network Congestion (Gradual Buildup)': np.int64(11), 'Faulty RF Filters (Temporal)': np.int64(9), 'Co-Channel Interference (Severe)': np.int64(8), 'Faulty Handover Algorithm (Too Fr

# Normal-session sizes

In [7]:
normal_only_sessions = [s for s in np.unique(session_id_by_row) if s not in anom_sessions]
for s in normal_only_sessions:
    print(f"Session {s}: {(session_id_by_row == s).sum()} rows")

Session 3: 3006 rows
Session 4: 3006 rows
Session 5: 3007 rows
Session 6: 3006 rows
Session 7: 3006 rows
Session 8: 3007 rows
Session 9: 3006 rows
Session 10: 3006 rows
Session 20: 309 rows
Session 21: 309 rows
Session 22: 309 rows
Session 23: 309 rows
Session 24: 309 rows
Session 25: 309 rows
Session 26: 309 rows
Session 27: 309 rows
Session 28: 309 rows
Session 29: 309 rows
Session 30: 309 rows
Session 31: 309 rows


# Build the final split

In [9]:
test_sessions = {13, 17, 1, 3, 4}  # anomalous: 13, 17, 1  |  normal: 3, 4

test_mask_new = np.isin(session_id_by_row, list(test_sessions))
train_mask_new = ~test_mask_new
train_idx_sessionsplit = np.where(train_mask_new)[0]
test_idx_sessionsplit = np.where(test_mask_new)[0]

print(f"New train size: {len(train_idx_sessionsplit):,}")
print(f"New test size: {len(test_idx_sessionsplit):,}")
print(f"New test anomaly rate: {100*anom_mask_full[test_mask_new].mean():.2f}%")

for cls in sorted(pd.Series(anomaly_type_full[anom_mask_full]).unique()):
    n_train = ((anomaly_type_full == cls) & train_mask_new).sum()
    n_test = ((anomaly_type_full == cls) & test_mask_new).sum()
    flag = "  <-- ZERO IN TEST" if n_test == 0 else ("  <-- ZERO IN TRAIN" if n_train == 0 else "")
    print(f"{cls}: train={n_train}, test={n_test}{flag}")

New train size: 25,688
New test size: 6,312
New test anomaly rate: 4.75%
Antenna Failure: train=98, test=9
Buffer Overflow (Gradual Buildup): train=117, test=21
Co-Channel Interference (Mild): train=126, test=20
Co-Channel Interference (Severe): train=84, test=30
Doppler Shift (Severe): train=77, test=9
Faulty Handover Algorithm (Too Frequent): train=45, test=19
Faulty RF Filters (Temporal): train=67, test=33
High Network Congestion (Gradual Buildup): train=56, test=37
High Network Congestion (Sudden Spike): train=37, test=11
Jamming: train=185, test=94
Resource Allocation Bugs: train=43, test=17


# Save the split

In [11]:
np.save(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), train_idx_sessionsplit)
np.save(os.path.join(DATA_DIR, "test_idx_sessionsplit.npy"), test_idx_sessionsplit)
print("Saved train_idx_sessionsplit.npy and test_idx_sessionsplit.npy to", DATA_DIR)

Saved train_idx_sessionsplit.npy and test_idx_sessionsplit.npy to /home/jovyan/kpi_rag/data


# Save the split

In [12]:
for fname in ["train_idx_sessionsplit.npy", "test_idx_sessionsplit.npy"]:
    fpath = os.path.join(DATA_DIR, fname)
    print(fname, "already exists:", os.path.isfile(fpath))

np.save(os.path.join(DATA_DIR, "train_idx_sessionsplit.npy"), train_idx_sessionsplit)
np.save(os.path.join(DATA_DIR, "test_idx_sessionsplit.npy"), test_idx_sessionsplit)
print("Saved train_idx_sessionsplit.npy and test_idx_sessionsplit.npy to", DATA_DIR)

train_idx_sessionsplit.npy already exists: True
test_idx_sessionsplit.npy already exists: True
Saved train_idx_sessionsplit.npy and test_idx_sessionsplit.npy to /home/jovyan/kpi_rag/data


# Rebuild feature/label arrays from the new indices

In [13]:
# Full-dataset feature matrix and labels (all 32,000 windows, aligned to df's row order)
X_features_full = np.load(os.path.join(DATA_DIR, "X_features.npy"), allow_pickle=True)  # adjust filename if different
y_labels_full = df["anomaly_type"].where(df["anomaly_status"] == "Anomaly", other="Normal").values  # adjust to match label convention

train_X_new = X_features_full[train_idx_sessionsplit]
test_X_new = X_features_full[test_idx_sessionsplit]
train_y_new = y_labels_full[train_idx_sessionsplit]
test_y_new = y_labels_full[test_idx_sessionsplit]

print("train_X_new:", train_X_new.shape)
print("test_X_new:", test_X_new.shape)
print("train_y_new:", train_y_new.shape)
print("test_y_new:", test_y_new.shape)

train_X_new: (25688, 582)
test_X_new: (6312, 582)
train_y_new: (25688,)
test_y_new: (6312,)


# Leakage check (cosine similarity, train vs. test)

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

# Sample for speed if the full pairwise comparison is too large
sample_train = train_X_new[np.random.RandomState(42).choice(len(train_X_new), min(2000, len(train_X_new)), replace=False)]
sample_test = test_X_new[np.random.RandomState(42).choice(len(test_X_new), min(2000, len(test_X_new)), replace=False)]

sim_matrix = cosine_similarity(sample_test, sample_train)
max_sim_per_test_row = sim_matrix.max(axis=1)

print("Max cosine similarity (test vs train) — summary:")
print(f"  Mean: {max_sim_per_test_row.mean():.4f}")
print(f"  95th percentile: {np.percentile(max_sim_per_test_row, 95):.4f}")
print(f"  Max: {max_sim_per_test_row.max():.4f}")
print(f"  Fraction >= 0.99: {(max_sim_per_test_row >= 0.99).mean():.4f}")

Max cosine similarity (test vs train) — summary:
  Mean: 0.9774
  95th percentile: 0.9988
  Max: 1.0000
  Fraction >= 0.99: 0.4405


In [16]:
# Confirm zero session overlap directly (the real guarantee we care about)
train_sessions_used = set(session_id_by_row[train_idx_sessionsplit])
test_sessions_used = set(session_id_by_row[test_idx_sessionsplit])
print("Session overlap between train and test:", train_sessions_used & test_sessions_used)

# Look specifically at near-DUPLICATE windows (very strict threshold), not just "similar"
near_duplicate_frac = (max_sim_per_test_row >= 0.999).mean()
exact_duplicate_frac = (max_sim_per_test_row >= 0.9999).mean()
print(f"Fraction of test rows with a near-duplicate (>=0.999) train row: {near_duplicate_frac:.4f}")
print(f"Fraction of test rows with a near-exact-duplicate (>=0.9999) train row: {exact_duplicate_frac:.4f}")

Session overlap between train and test: set()
Fraction of test rows with a near-duplicate (>=0.999) train row: 0.0405
Fraction of test rows with a near-exact-duplicate (>=0.9999) train row: 0.0065


# saving the arrays we built

In [18]:
np.save(os.path.join(DATA_DIR, "train_X_sessionsplit.npy"), train_X_new)
np.save(os.path.join(DATA_DIR, "test_X_sessionsplit.npy"), test_X_new)
np.save(os.path.join(DATA_DIR, "train_y_sessionsplit.npy"), train_y_new)
np.save(os.path.join(DATA_DIR, "test_y_sessionsplit.npy"), test_y_new)
print("Saved rebuilt session-split feature/label arrays to", DATA_DIR)

Saved rebuilt session-split feature/label arrays to /home/jovyan/kpi_rag/data


In [19]:
np.save(os.path.join(DATA_DIR, "session_id_by_row.npy"), session_id_by_row)